<a href="https://colab.research.google.com/github/Muskan910/hinglish-assistant/blob/main/notebooks/00_setup_verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth wandb huggingface_hub


In [ ]:
# Add this as the first cell of every Colab notebook
import os
os.environ["TQDM_DISABLE"] = "1"

# And suppress HuggingFace's progress bars specifically
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import wandb

login(token=userdata.get('HF_TOKEN'))
wandb.login(key=userdata.get('WANDB_API_KEY'))
print("Auth done")


In [ ]:
## Load Qwen2.5-3B in 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print(f"Loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Bhai weekend pe Bangalore mein kya karein?"}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to("cuda")
outputs = model.generate(inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Cell 5: Verify W&B works
run = wandb.init(project="hinglish-assistant", name="setup-test")
wandb.log({"setup_complete": 1, "gpu_mem_gb": torch.cuda.memory_allocated() / 1e9})
run.finish()
print("Setup verified. Check wandb.ai for the run.")